1. 라이브러리 설치

In [2]:
# 1. 훈련에 필요한 핵심 라이브러리 설치
!pip install transformers datasets accelerate

# 2. 라이브러리 임포트
import torch
import pandas as pd
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split # 훈련/검증용 분리
from transformers import AutoTokenizer, AutoModelForSequenceClassification

print("\n--- 라이브러리 임포트 완료 ---")

# 3. (★중요★) GPU가 활성화되었는지 확인
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ 현재 사용 중인 디바이스: {device}")

if device.type == 'cpu':
    print("🚨 [경고] GPU가 활성화되지 않았습니다!")
    print("   [런타임] > [런타임 유형 변경]에서 'T4 GPU'로 설정하세요.")


--- 라이브러리 임포트 완료 ---
✅ 현재 사용 중인 디바이스: cuda


2. '교과서' 로드 및 훈련/검증 세트 분리

In [3]:
# 1. (★중요★) 업로드한 '교과서'를 Pandas로 불러옵니다.
# (파일이 'snv_train_dataset.csv'가 맞는지 확인하세요!)
try:
    full_df = pd.read_csv('snv_train_dataset.csv')
    print(f"✅ 'snv_train_dataset.csv' 로드 성공! (총 {len(full_df)}개 데이터)")
except FileNotFoundError:
    print("🚨 [에러] 'snv_train_dataset.csv' 파일을 찾을 수 없습니다!")
    print("   왼쪽 파일 탐색기(📁)에 파일을 업로드했는지 확인하세요.")

# 2. Pandas DataFrame을 Hugging Face 'Dataset' 객체로 변환
dataset = Dataset.from_pandas(full_df)

# 3. 훈련용(Train) 90% / 평가용(Evaluation) 10%로 분리
# (test_size=0.1 이 10%를 의미. random_state=42는 재현성을 위함)
train_eval_split = dataset.train_test_split(test_size=0.1, seed=42)

# 4. 최종 데이터셋 딕셔너리 생성
# 'train' 키에는 훈련용 데이터가,
# 'test' 키에는 평가용 데이터가 들어갑니다.
ds = DatasetDict({
    'train': train_eval_split['train'],
    'eval': train_eval_split['test']
})

print("\n--- 데이터 분리 완료 ---")
print(f"훈련용 데이터: {len(ds['train'])} 개")
print(f"평가용 데이터: {len(ds['eval'])} 개")
print("\n[데이터 샘플]")
print(ds['train'][0]) # 0번째 훈련 데이터 샘플 출력

✅ 'snv_train_dataset.csv' 로드 성공! (총 261931개 데이터)

--- 데이터 분리 완료 ---
훈련용 데이터: 235737 개
평가용 데이터: 26194 개

[데이터 샘플]
{'ref_seq': 'TTACTTCGTCTATCTGTCTGGAAATGGTACTGCTCTTCTTTGGAATGGTGTTTCATCATCTGTACATCAAAAGATTTAACTGCATGATTACCACTGTTTCTTAAACCTTCGTGACTTCTTTACAGCTCAGTTCACCTATGTCTCTTGTTTCTAGGGCACAACCATATTAACTTCCCTCACTTCTGTGCTACATGACAACAAAGAATTTCCCAACCCAGAGATGTTTGACCCTCGTCACTTTCTGGATGAAGGTGGAAATTTTAAGAAAAGTAACTACTTCATGCCTTTCTCAGCAGGTAATATAAATTTATTTCCCTTTGTGTTTCAGGGTACAAGATAACTTTTTTGATCAGTTGGAACTTACATGTGCCTTCTCTGCAGTGGTACAGTTACTCTTTGTACATGATCAAGAGCACTGTTCTGAATGCCTGTGTTTTCTCCGCTGGTGATACATCCTCATTATTCGGCCAGATTAGTGGGTTTTGGAGAATTAATCCAATTCTTCCAAATT', 'var_seq': 'TTACTTCGTCTATCTGTCTGGAAATGGTACTGCTCTTCTTTGGAATGGTGTTTCATCATCTGTACATCAAAAGATTTAACTGCATGATTACCACTGTTTCTTAAACCTTCGTGACTTCTTTACAGCTCAGTTCACCTATGTCTCTTGTTTCTAGGGCACAACCATATTAACTTCCCTCACTTCTGTGCTACATGACAACAAAGAATTTCCCAACCCAGAGATGTTTGACCCTCGTCACTTTCTGGATGAAGGTGGCAATTTTAAGAAAAGTAACTACTTCATGCCTTTCTCAGCAGGTAATATAAATTTATTTCCCTTTGTGTTTCAGGGTACAAGATAACTTTTTTG

3. 토큰화

In [4]:
# [★3단계 최종 수정본★ - 이 코드를 사용하세요]

from transformers import AutoTokenizer

# 1. 토크나이저를 불러옵니다.
MODEL_ID = "InstaDeepAI/nucleotide-transformer-v2-500m-multi-species"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

# 2. [SEP], [EOS] 토큰을 *추가*합니다.
print(f"수정 전 토크나이저 크기: {len(tokenizer)}")
tokenizer.add_special_tokens({
    'sep_token': '[SEP]',
    'eos_token': '[EOS]'
})
print(f"수정 후 토크나이저 크기: {len(tokenizer)}")

print(f"\n✅ 토크나이저 로드 및 토큰 추가 완료!")

# 3. '서열 쌍'을 토큰화하는 함수를 정의합니다.
def tokenize_sequence_pair(examples):
    return tokenizer(
        examples['ref_seq'],
        examples['var_seq'],
        truncation=True,
        max_length=512
    )

print("\n--- 토큰화 함수 정의 완료 ---")
print("데이터셋 전체에 토큰화를 적용합니다. (몇 분 정도 소요될 수 있습니다...)")

# 4. 26만 개 전체 데이터셋(ds)에 토큰화 함수를 적용(map)합니다.
tokenized_ds = ds.map(tokenize_sequence_pair, batched=True)

print("✅ 토큰화 완료!")

# 5. 'label' 컬럼 이름을 'labels'로 변경합니다.
tokenized_ds = tokenized_ds.rename_column("label", "labels")

# ★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
# [수정된 부분]
# 존재하지 않는 '__index_level_0__' 컬럼을 '청소 목록'에서 제외합니다.
tokenized_ds = tokenized_ds.remove_columns(['ref_seq', 'var_seq'])
# ★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★

print("--- 데이터셋 전처리(Tokenizing) 완료 ---")
print("최종 데이터셋 구조:")
print(tokenized_ds)
print("\n[토큰화된 샘플 데이터]")
print(tokenized_ds['train'][0])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


수정 전 토크나이저 크기: 4107
수정 후 토크나이저 크기: 4109

✅ 토크나이저 로드 및 토큰 추가 완료!

--- 토큰화 함수 정의 완료 ---
데이터셋 전체에 토큰화를 적용합니다. (몇 분 정도 소요될 수 있습니다...)


Map:   0%|          | 0/235737 [00:00<?, ? examples/s]

Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.


Map:   0%|          | 0/26194 [00:00<?, ? examples/s]

✅ 토큰화 완료!
--- 데이터셋 전처리(Tokenizing) 완료 ---
최종 데이터셋 구조:
DatasetDict({
    train: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 235737
    })
    eval: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 26194
    })
})

[토큰화된 샘플 데이터]
{'labels': 0, 'input_ids': [3, 1323, 2922, 1660, 1990, 2008, 1951, 1629, 3109, 1884, 396, 1870, 1542, 3162, 638, 1818, 2605, 1387, 47, 1762, 2411, 1172, 1595, 2218, 1900, 1499, 2373, 2184, 2123, 156, 2664, 1443, 3662, 1830, 2066, 352, 2096, 823, 3426, 2721, 1579, 1666, 1813, 1990, 1366, 3081, 1066, 2407, 3739, 2452, 982, 1094, 1355, 1707, 1915, 1605, 1161, 271, 1373, 403, 1992, 1319, 3567, 1645, 2277, 1171, 1183, 1496, 460, 212, 635, 2503, 3747, 3420, 1727, 3959, 544, 1562, 1476, 2251, 901, 1381, 779, 110, 363, 2567, 4103, 4108, 1323, 2922, 1660, 1990, 2008, 1951, 1629, 3109, 1884, 396, 1870, 1542, 3162, 638, 1818, 2605, 1387, 47, 1762, 2411, 1172, 1595, 2218, 1900, 1499, 2373, 2

4. '분류용' 모델 로드

In [5]:
# [★최종 수정본 4단계★ - 이 코드를 사용하세요]

from transformers import AutoModelForSequenceClassification

print(f"사전 학습된 모델({MODEL_ID})을 로드합니다...")

# 1. (이전과 동일) 분류용 모델 로드
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID,
    num_labels=2,
    trust_remote_code=True
)

# ★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
# [최종 수정된 부분]
# 토크나이저에 2개의 새 토큰을 추가했으므로,
# 모델의 '단어장(Embedding Layer)' 크기도 2개 늘려줘야 합니다.
# 그렇지 않으면 모델과 토크나이저의 크기가 안 맞아 에러가 납니다.
print(f"모델 임베딩 크기 리사이징 전: {model.get_input_embeddings().weight.size(0)}")
model.resize_token_embeddings(len(tokenizer))
print(f"모델 임베딩 크기 리사이징 후: {model.get_input_embeddings().weight.size(0)}")
# ★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★

# 2. 모델을 GPU로 보냅니다.
model = model.to(device)

print("\n✅ '분류용' 모델 로드 및 리사이징 완료!")
print("--- [Phase 4]의 모든 준비가 끝났습니다. ---")

사전 학습된 모델(InstaDeepAI/nucleotide-transformer-v2-500m-multi-species)을 로드합니다...


Some weights of the model checkpoint at InstaDeepAI/nucleotide-transformer-v2-500m-multi-species were not used when initializing EsmForSequenceClassification: ['lm_head.bias', 'lm_head.decoder.weight', 'lm_head.dense.bias', 'lm_head.dense.weight', 'lm_head.layer_norm.bias', 'lm_head.layer_norm.weight']
- This IS expected if you are initializing EsmForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing EsmForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of EsmForSequenceClassification were not initialized from the model checkpoint at InstaDeepAI/nucleotide-transformer-v2-500m-multi-species and are newly initialized: ['classifier.dense.bias', 

모델 임베딩 크기 리사이징 전: 4107
모델 임베딩 크기 리사이징 후: 4109

✅ '분류용' 모델 로드 및 리사이징 완료!
--- [Phase 4]의 모든 준비가 끝났습니다. ---


5. 모델 훈련 (Fine-tuning)

In [6]:
!pip install evaluate

In [11]:
# [★5단계 수정본★ - 이 코드를 사용하세요]

from transformers import TrainingArguments, Trainer
import numpy as np
import evaluate # (이 코드를 위해 !pip install evaluate 를 먼저 실행해야 합니다)

print("--- 훈련(Fine-tuning)을 시작합니다 ---")
print(f"훈련 데이터: {len(tokenized_ds['train'])} 개")
print(f"평가 데이터: {len(tokenized_ds['eval'])} 개")

# 1. 훈련 규칙 (TrainingArguments) 정의
training_args = TrainingArguments(
    output_dir="./results",

    # --- 훈련 스케줄 ---
    num_train_epochs=1,
    per_device_train_batch_size=8,

    # ★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
    # [수정된 부분]
    # 'evaluation_strategy' (신버전) -> 'eval_strategy' (구버전)로 이름 변경
    eval_strategy="steps",
    # ★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★

    eval_steps=2000,

    # --- 로깅 및 저장 ---
    logging_steps=500,
    save_strategy="steps", # 'save_strategy'는 이름이 동일합니다.
    save_steps=2000,

    # --- 성능 최적화 ---
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    fp16=True,

    report_to="none",
)

# 2. '모의고사 채점기' 정의
accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_preds):
    logits, labels = eval_preds
    predictions = np.argmax(logits, axis=-1)

    return accuracy_metric.compute(
        predictions=predictions,
        references=labels
    )

# 3. '트레이너' 객체 생성
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds['train'],
    eval_dataset=tokenized_ds['eval'],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

print("\n--- '트레이너' 준비 완료. GPU로 훈련을 시작합니다! ---")
print("   (T4 GPU 기준 30분~1시간 이상 소요됩니다...)")

# 4. 훈련 시작!
trainer.train()

print("\n🎉🎉🎉 훈련(Fine-tuning)이 완료되었습니다! 🎉🎉🎉")

--- 훈련(Fine-tuning)을 시작합니다 ---
훈련 데이터: 235737 개
평가 데이터: 26194 개


/tmp/ipython-input-2078107263.py:53: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(



--- '트레이너' 준비 완료. GPU로 훈련을 시작합니다! ---
   (T4 GPU 기준 30분~1시간 이상 소요됩니다...)


OutOfMemoryError: CUDA out of memory. Tried to allocate 20.00 MiB. GPU 0 has a total capacity of 14.74 GiB of which 20.12 MiB is free. Process 292610 has 14.72 GiB memory in use. Of the allocated memory 14.31 GiB is allocated by PyTorch, and 287.49 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [13]:
from google.colab import drive
import os

# Drive 강제 재연결 (이전에 오류가 발생했더라도 다시 시도합니다)
drive.mount('/content/gdrive', force_remount=True)

# Drive에 저장할 최종 경로 설정
# (이 경로에 'checkpoint-4000' 폴더가 통째로 복사됩니다)
DRIVE_DEST_DIR = '/content/gdrive/My Drive/DACON/MAI/checkpoints/checkpoint-4000'

# Colab 임시 저장소에 있는 원본 파일 경로
COLAB_SOURCE_DIR = './results/checkpoint-4000'

print(f"Drive 목적지 폴더: {DRIVE_DEST_DIR}")

Mounted at /content/gdrive
Drive 목적지 폴더: /content/gdrive/My Drive/DACON/MAI/checkpoints/checkpoint-4000


In [14]:
# Drive에 목적지 폴더가 없으면 새로 생성합니다.
!mkdir -p "{DRIVE_DEST_DIR}"

# 🌟🌟🌟 가장 중요한 복사 명령 🌟🌟🌟
# 'cp -r' 명령어로 원본 폴더 전체를 Drive로 재귀적(r)으로 복사합니다.
print(f"복사 시작: {COLAB_SOURCE_DIR} -> Drive...")
!cp -r "{COLAB_SOURCE_DIR}/"* "{DRIVE_DEST_DIR}/"

# 복사 완료 후 Drive에서 파일 목록 확인
print("\n✅ Drive 복사 완료! (Drive에서 파일 확인)")
!ls -lh "{DRIVE_DEST_DIR}"

복사 시작: ./results/checkpoint-4000 -> Drive...

✅ Drive 복사 완료! (Drive에서 파일 확인)
total 5.6G
-rw------- 1 root root   37 Nov 11 02:15 added_tokens.json
-rw------- 1 root root 1.1K Nov 11 02:15 config.json
-rw------- 1 root root  15K Nov 11 02:15 esm_config.py
-rw------- 1 root root 1.9G Nov 11 02:15 model.safetensors
-rw------- 1 root root 3.7G Nov 11 02:17 optimizer.pt
-rw------- 1 root root  15K Nov 11 02:17 rng_state.pth
-rw------- 1 root root 1.4K Nov 11 02:17 scaler.pt
-rw------- 1 root root 1.5K Nov 11 02:17 scheduler.pt
-rw------- 1 root root  377 Nov 11 02:17 special_tokens_map.json
-rw------- 1 root root 1.3K Nov 11 02:17 tokenizer_config.json
-rw------- 1 root root 2.7K Nov 11 02:17 trainer_state.json
-rw------- 1 root root 5.7K Nov 11 02:17 training_args.bin
-rw------- 1 root root  29K Nov 11 02:17 vocab.txt
